# ODSB-16780: Postback Analysis for `com.netmarble.sololv`

**Date**: 2026-03-09  
**App**: Solo Leveling ARISE (Android, `com.netmarble.sololv`)  
**MMP**: AppsFlyer  
**Suspicious campaigns**: `Jl5NUKiajhIyKXQK`, `fISFGcvrsN7uXPoR`, `nqWMPgcAgXjZgEM8`

## Key Question

**Is the missing revenue limited to specific campaign-attributed data, or does it also affect unattributed postbacks?**

- If raw postbacks (attributed + unattributed) also show $0 revenue → **MMP-side issue** (AppsFlyer not sending revenue values)
- If raw postbacks have revenue but attributed data shows $0 → **attribution/processing issue** on Moloco side

## Tables
- **Attributed**: `moloco-ae-view.athena.fact_dsp_all` (campaign-attributed events)
- **All postbacks (attributed + unattributed)**: `focal-elf-631.df_accesslog.pb` (~1TB/day, use with care)

In [74]:
#@title Environment Setup
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

client = bigquery.Client(project='moloco-ods')

def run_query(query, label=''):
    """Run a BQ query and return DataFrame. Print row count."""
    try:
        df = client.query(query).result().to_dataframe()
        status = f'✅ {label}: {len(df)} rows' if len(df) > 0 else f'⚠️ {label}: 0 rows — check needed'
        print(status)
        return df
    except Exception as e:
        print(f'❌ {label}: {e}')
        return pd.DataFrame()

print("Setup complete.")

Setup complete.


## 1. All Attributed Events — Name, Type, Revenue

Check all event types/names for `com.netmarble.sololv` in the attributed postback table, with revenue indicators.

In [75]:
df_events = run_query("""
SELECT
  cv.pb.event.name AS event_name,
  COUNT(*) AS event_count,
  COUNTIF(cv.revenue_usd.amount > 0) AS events_with_revenue,
  COUNTIF(cv.revenue_usd.amount = 0 OR cv.revenue_usd.amount IS NULL) AS events_without_revenue,
  ROUND(SUM(IFNULL(cv.revenue_usd.amount, 0)), 2) AS total_revenue_usd
FROM `focal-elf-631.prod_stream_view.cv`
WHERE cv.pb.app.bundle = 'com.netmarble.sololv'
  AND DATE(timestamp) BETWEEN '2026-02-28' AND '2026-03-08'
  AND cv.pb.moloco.attributed = true
GROUP BY 1
ORDER BY event_count DESC
""", '1. All Attributed Events (cv)')

if not df_events.empty:
    print(f"Total event names: {len(df_events)}")
    print(f"\n=== Events WITH revenue ===")
    display(df_events[df_events['total_revenue_usd'] > 0])
    print(f"\n=== All events (top 30 by count) ===")
    display(df_events.head(30))

✅ 1. All Attributed Events (cv): 45 rows
Total event names: 45

=== Events WITH revenue ===


,event_name,event_count,events_with_revenue,events_without_revenue,total_revenue_usd
37,revenue,1893,1827,66,11064.53



=== All events (top 30 by count) ===


,event_name,event_count,events_with_revenue,events_without_revenue,total_revenue_usd
0,af_app_opened,288350,0,288350,0.0
1,visit_shop,198767,0,198767,0.0
2,login,190408,0,190408,0.0
3,gacha_normal_x10,123612,0,123612,0.0
4,lvup_hunter_1st,102393,0,102393,0.0
5,enhance_artifact_1st,87797,0,87797,0.0
6,cdn_download_finished,79096,0,79096,0.0
7,lvup_weapon_1st,70353,0,70353,0.0
8,lvup_skill_1st,53566,0,53566,0.0
9,gacha_limited_x10,48862,0,48862,0.0


## 2. Campaign-Level Attributed Revenue Events

Daily event count and revenue amount for revenue events, broken down by campaign.

In [84]:
df_campaign_rev = run_query("""
SELECT
  cv.pb.moloco.campaign_id AS campaign_id,
  DATE(timestamp) AS date,
  COUNT(*) AS revenue_events,
  COUNTIF(cv.revenue_usd.amount > 0) AS events_with_revenue,
  ROUND(SUM(IFNULL(cv.revenue_usd.amount, 0)), 4) AS pb_revenue
FROM `focal-elf-631.prod_stream_view.cv`
WHERE cv.pb.app.bundle = 'com.netmarble.sololv'
  AND DATE(timestamp) BETWEEN '2026-02-28' AND '2026-03-08'
  AND cv.pb.event.name = 'revenue'
  AND cv.pb.moloco.attributed = true
GROUP BY 1, 2
ORDER BY campaign_id, date
""", '2. Campaign-Level Revenue Events (cv)')

if not df_campaign_rev.empty:
    print(f"Unique campaigns with revenue events: {df_campaign_rev['campaign_id'].nunique()}")
    display(df_campaign_rev)

✅ 2. Campaign-Level Revenue Events (cv): 65 rows
Unique campaigns with revenue events: 8


,campaign_id,date,revenue_events,events_with_revenue,pb_revenue
0,BNnzFJ1BQ6sRK9C3,2026-02-28,48,48,347.6458
1,BNnzFJ1BQ6sRK9C3,2026-03-01,29,29,158.9461
2,BNnzFJ1BQ6sRK9C3,2026-03-02,19,19,92.7195
3,BNnzFJ1BQ6sRK9C3,2026-03-03,20,20,101.8394
4,BNnzFJ1BQ6sRK9C3,2026-03-04,37,37,147.1363
5,BNnzFJ1BQ6sRK9C3,2026-03-05,18,18,137.0382
6,BNnzFJ1BQ6sRK9C3,2026-03-06,45,45,395.3675
7,BNnzFJ1BQ6sRK9C3,2026-03-07,23,23,88.3374
8,BNnzFJ1BQ6sRK9C3,2026-03-08,23,23,206.8528
9,BnyGc57kJEoPrxdW,2026-02-28,97,97,660.2084


In [86]:
# Pivot: campaign x date — revenue event count vs actual revenue value
if not df_campaign_rev.empty:
    pivot_event_count = df_campaign_rev.pivot_table(
        index='campaign_id', columns='date', values='revenue_events', aggfunc='sum', fill_value=0
    )
    print("=== Revenue Event Count by Campaign x Date ===")
    display(pivot_event_count)

    pivot_with_rev = df_campaign_rev.pivot_table(
        index='campaign_id', columns='date', values='events_with_revenue', aggfunc='sum', fill_value=0
    )
    print("\n=== Revenue Events WITH Actual Revenue Value by Campaign x Date ===")
    display(pivot_with_rev)

    pivot_rev = df_campaign_rev.pivot_table(
        index='campaign_id', columns='date', values='pb_revenue', aggfunc='sum', fill_value=0
    )
    print("\n=== PB Revenue ($) by Campaign x Date ===")
    display(pivot_rev)

    # Per campaign x date flag
    print("\n=== Campaign Flag by Date: Revenue Events vs Events with Actual Revenue ===")
    flag_df = df_campaign_rev.copy()
    flag_df['pct_with_revenue'] = (flag_df['events_with_revenue'] / flag_df['revenue_events'] * 100).round(1)
    flag_df['flag'] = flag_df.apply(
        lambda r: 'ALL MISSING' if r['events_with_revenue'] == 0
        else 'PARTIAL' if r['events_with_revenue'] < r['revenue_events']
        else 'OK', axis=1
    )
    display(flag_df[['campaign_id', 'date', 'revenue_events', 'events_with_revenue', 'pct_with_revenue', 'pb_revenue', 'flag']].sort_values(['campaign_id', 'date']))

    # Flag pivot: campaign x date
    print("\n=== Flag Pivot: Campaign x Date ===")
    flag_pivot = flag_df.pivot_table(index='campaign_id', columns='date', values='flag', aggfunc='first', fill_value='—')
    display(flag_pivot)

=== Revenue Event Count by Campaign x Date ===


date,2026-02-28,2026-03-01,2026-03-02,2026-03-03,2026-03-04,2026-03-05,2026-03-06,2026-03-07,2026-03-08
campaign_id,,,,,,,,,
BNnzFJ1BQ6sRK9C3,48,29,19,20,37,18,45,23,23
BnyGc57kJEoPrxdW,97,65,40,48,50,44,44,44,44
Jl5NUKiajhIyKXQK,0,0,2,4,0,5,1,7,9
PcXr8D0MsfIKFEzC,41,53,10,25,15,9,11,8,25
VBnWxKikoKxWVkVa,113,93,92,48,61,26,47,39,59
ZQUR6yZ4jwpXrTgZ,36,28,25,27,39,21,27,27,26
fISFGcvrsN7uXPoR,0,0,1,0,2,5,1,6,10
nqWMPgcAgXjZgEM8,0,7,1,2,6,5,15,19,16



=== Revenue Events WITH Actual Revenue Value by Campaign x Date ===


date,2026-02-28,2026-03-01,2026-03-02,2026-03-03,2026-03-04,2026-03-05,2026-03-06,2026-03-07,2026-03-08
campaign_id,,,,,,,,,
BNnzFJ1BQ6sRK9C3,48,29,19,20,37,18,45,23,23
BnyGc57kJEoPrxdW,97,65,40,48,50,44,44,44,44
Jl5NUKiajhIyKXQK,0,0,2,4,0,0,0,4,2
PcXr8D0MsfIKFEzC,41,53,10,25,15,9,11,8,25
VBnWxKikoKxWVkVa,113,93,92,48,61,26,47,39,59
ZQUR6yZ4jwpXrTgZ,36,28,25,27,39,21,27,27,26
fISFGcvrsN7uXPoR,0,0,1,0,2,3,1,1,5
nqWMPgcAgXjZgEM8,0,7,1,2,0,2,9,6,6



=== PB Revenue ($) by Campaign x Date ===


date,2026-02-28,2026-03-01,2026-03-02,2026-03-03,2026-03-04,2026-03-05,2026-03-06,2026-03-07,2026-03-08
campaign_id,,,,,,,,,
BNnzFJ1BQ6sRK9C3,347.6458,158.9461,92.7195,101.8394,147.1363,137.0382,395.3675,88.3374,206.8528
BnyGc57kJEoPrxdW,660.2084,502.2712,221.1941,224.1986,290.6267,219.2946,204.5085,244.7410,198.1414
Jl5NUKiajhIyKXQK,0.0000,0.0000,44.4407,48.4416,0.0000,0.0000,0.0000,14.1458,8.8332
PcXr8D0MsfIKFEzC,218.3334,311.8925,90.4167,164.0033,108.7440,41.1114,69.8676,51.8681,148.9303
VBnWxKikoKxWVkVa,785.0974,627.1218,657.7230,160.8458,436.8681,120.6015,149.7499,313.9008,341.3544
ZQUR6yZ4jwpXrTgZ,270.2975,167.5800,133.0395,155.0615,200.1615,120.9687,146.1186,139.1793,116.4887
fISFGcvrsN7uXPoR,0.0000,0.0000,1.7776,0.0000,6.2669,2.3764,3.9075,5.8941,22.1418
nqWMPgcAgXjZgEM8,0.0000,68.6013,0.6060,4.5520,0.0000,11.8819,42.9965,22.5329,66.7394



=== Campaign Flag by Date: Revenue Events vs Events with Actual Revenue ===


,campaign_id,date,revenue_events,events_with_revenue,pct_with_revenue,pb_revenue,flag
0,BNnzFJ1BQ6sRK9C3,2026-02-28,48,48,100.0,347.6458,OK
1,BNnzFJ1BQ6sRK9C3,2026-03-01,29,29,100.0,158.9461,OK
2,BNnzFJ1BQ6sRK9C3,2026-03-02,19,19,100.0,92.7195,OK
3,BNnzFJ1BQ6sRK9C3,2026-03-03,20,20,100.0,101.8394,OK
4,BNnzFJ1BQ6sRK9C3,2026-03-04,37,37,100.0,147.1363,OK
5,BNnzFJ1BQ6sRK9C3,2026-03-05,18,18,100.0,137.0382,OK
6,BNnzFJ1BQ6sRK9C3,2026-03-06,45,45,100.0,395.3675,OK
7,BNnzFJ1BQ6sRK9C3,2026-03-07,23,23,100.0,88.3374,OK
8,BNnzFJ1BQ6sRK9C3,2026-03-08,23,23,100.0,206.8528,OK
9,BnyGc57kJEoPrxdW,2026-02-28,97,97,100.0,660.2084,OK



=== Flag Pivot: Campaign x Date ===


date,2026-02-28,2026-03-01,2026-03-02,2026-03-03,2026-03-04,2026-03-05,2026-03-06,2026-03-07,2026-03-08
campaign_id,,,,,,,,,
BNnzFJ1BQ6sRK9C3,OK,OK,OK,OK,OK,OK,OK,OK,OK
BnyGc57kJEoPrxdW,OK,OK,OK,OK,OK,OK,OK,OK,OK
Jl5NUKiajhIyKXQK,—,—,OK,OK,—,ALL MISSING,ALL MISSING,PARTIAL,PARTIAL
PcXr8D0MsfIKFEzC,OK,OK,OK,OK,OK,OK,OK,OK,OK
VBnWxKikoKxWVkVa,OK,OK,OK,OK,OK,OK,OK,OK,OK
ZQUR6yZ4jwpXrTgZ,OK,OK,OK,OK,OK,OK,OK,OK,OK
fISFGcvrsN7uXPoR,—,—,OK,—,OK,PARTIAL,OK,PARTIAL,PARTIAL
nqWMPgcAgXjZgEM8,—,OK,OK,OK,ALL MISSING,PARTIAL,PARTIAL,PARTIAL,PARTIAL


✅ 2b. nqWMPgcAgXjZgEM8 Events (cv): 366 rows

=== Event summary (aggregated across dates) ===


,total_events,total_with_revenue,total_revenue,days_seen
event_name,,,,
visit_shop,6680,0,0.00,9
af_app_opened,6552,0,0.00,9
gacha_normal_x10,6120,0,0.00,9
login,5070,0,0.00,9
cdn_download_finished,4573,0,0.00,9
lvup_hunter_1st,4421,0,0.00,9
reengagement,3322,0,0.00,9
lvup_weapon_1st,2504,0,0.00,8
gacha_limited_x10,2271,0,0.00,9


## 3. Campaign Configuration — Goal, Tracking Links, KPI Setup

Check campaign details and tracking link configuration for the 3 suspicious campaigns.

In [79]:
df_campaign_details = run_query("""
WITH core AS (
  SELECT
    campaign_id,
    ANY_VALUE(campaign.title) AS title,
    ANY_VALUE(campaign.goal) AS goal,
    ANY_VALUE(campaign.os) AS os,
    ANY_VALUE(campaign.country) AS country,
    MIN(date_utc) AS first_date,
    MAX(date_utc) AS last_date,
    ROUND(SUM(gross_spend_usd), 2) AS total_spend,
    SUM(impressions) AS total_impressions,
    SUM(installs) AS total_installs
  FROM `moloco-ae-view.athena.fact_dsp_core`
  WHERE campaign_id IN ('Jl5NUKiajhIyKXQK', 'fISFGcvrsN7uXPoR', 'nqWMPgcAgXjZgEM8')
    AND date_utc BETWEEN '2026-02-01' AND '2026-03-09'
  GROUP BY 1
),
rev AS (
  SELECT
    cv.pb.moloco.campaign_id AS campaign_id,
    COUNT(*) AS total_revenue_events,
    COUNTIF(cv.revenue_usd.amount > 0) AS events_with_revenue,
    ROUND(SUM(IFNULL(cv.revenue_usd.amount, 0)), 2) AS total_revenue_usd
  FROM `focal-elf-631.prod_stream_view.cv`
  WHERE cv.pb.moloco.campaign_id IN ('Jl5NUKiajhIyKXQK', 'fISFGcvrsN7uXPoR', 'nqWMPgcAgXjZgEM8')
    AND DATE(timestamp) BETWEEN '2026-02-01' AND '2026-03-09'
    AND cv.pb.event.name = 'revenue'
    AND cv.pb.moloco.attributed = true
  GROUP BY 1
)
SELECT c.*, r.total_revenue_events, r.events_with_revenue, r.total_revenue_usd
FROM core c
LEFT JOIN rev r USING (campaign_id)
""", '3. Campaign Details')

if not df_campaign_details.empty:
    display(df_campaign_details)

✅ 3. Campaign Details: 3 rows


,campaign_id,title,goal,os,country,first_date,last_date,total_spend,total_impressions,total_installs,total_revenue_events,events_with_revenue,total_revenue_usd
0,Jl5NUKiajhIyKXQK,WISEBIRDS_[XRTX]sololv_update_moloco_WW_And_tR...,OPTIMIZE_ROAS_FOR_APP_RE,ANDROID,CIV,2026-02-28,2026-03-09,8690.730000000,13396602,60,30,13,119.75
1,fISFGcvrsN7uXPoR,WISEBIRDS_[XRTX]sololv_update_moloco_KR_And_tR...,OPTIMIZE_ROAS_FOR_APP_RE,ANDROID,KOR,2026-02-28,2026-03-09,20880.360000000,3868361,1,33,19,51.20
2,nqWMPgcAgXjZgEM8,WISEBIRDS_[XRTX]sololv_update_moloco_US_And_AC...,OPTIMIZE_CPA_FOR_APP_RE,ANDROID,USA,2026-02-28,2026-03-09,13306.680000000,2856088,2,82,33,217.91


In [80]:
df_event_config = run_query("""
SELECT
  cv.pb.moloco.campaign_id AS campaign_id,
  cv.pb.event.name AS event_name,
  COUNT(*) AS event_count,
  COUNTIF(cv.revenue_usd.amount > 0) AS with_revenue,
  ROUND(SUM(IFNULL(cv.revenue_usd.amount, 0)), 4) AS total_revenue
FROM `focal-elf-631.prod_stream_view.cv`
WHERE cv.pb.moloco.campaign_id IN ('Jl5NUKiajhIyKXQK', 'fISFGcvrsN7uXPoR', 'nqWMPgcAgXjZgEM8')
  AND DATE(timestamp) BETWEEN '2026-02-28' AND '2026-03-08'
  AND cv.pb.moloco.attributed = true
GROUP BY 1, 2
HAVING event_count > 0
ORDER BY campaign_id, event_count DESC
""", '3b. Event Config per Campaign (cv)')

if not df_event_config.empty:
    display(df_event_config)

✅ 3b. Event Config per Campaign (cv): 135 rows


,campaign_id,event_name,event_count,with_revenue,total_revenue
0,Jl5NUKiajhIyKXQK,af_app_opened,210283,0,0.0
1,Jl5NUKiajhIyKXQK,login,130325,0,0.0
2,Jl5NUKiajhIyKXQK,visit_shop,91022,0,0.0
3,Jl5NUKiajhIyKXQK,gacha_normal_x10,67711,0,0.0
4,Jl5NUKiajhIyKXQK,cdn_download_finished,56497,0,0.0
...,...,...,...,...,...
130,nqWMPgcAgXjZgEM8,clear_chpt4,56,0,0.0
131,nqWMPgcAgXjZgEM8,checkin28d_28d,50,0,0.0
132,nqWMPgcAgXjZgEM8,lvachieved_user_10,34,0,0.0
133,nqWMPgcAgXjZgEM8,lvachieved_user_15,31,0,0.0


## 3b. Raw Postback: Attributed vs Unattributed Revenue (KEY SECTION)

Compare revenue events in `df_accesslog.pb` **before** Moloco attribution processing.  
If unattributed postbacks also have $0 revenue → MMP-side problem.  
If only attributed postbacks show $0 → attribution/processing issue.

In [81]:
df_pb_revenue = run_query("""
SELECT
  DATE(timestamp) AS date,
  attribution.attributed,
  CASE
    WHEN event.revenue_usd.amount > 0 THEN 'has_revenue'
    WHEN event.revenue_usd.amount = 0 THEN 'zero_revenue'
    ELSE 'null_revenue'
  END AS revenue_status,
  COUNT(*) AS event_count,
  ROUND(SUM(IFNULL(event.revenue_usd.amount, 0)), 4) AS total_revenue_usd
FROM `focal-elf-631.df_accesslog.pb`
WHERE DATE(timestamp) BETWEEN '2026-03-01' AND '2026-03-08'
  AND app.bundle = 'com.netmarble.sololv'
  AND event.name = 'revenue'
GROUP BY 1, 2, 3
ORDER BY date, attributed DESC, revenue_status
""", '3b. Raw PB Attributed vs Unattributed')

if not df_pb_revenue.empty:
    display(df_pb_revenue)

    print("\n=== Pivot: Daily Revenue by Attribution Status ===")
    pivot = df_pb_revenue.groupby(['date', 'attributed']).agg(
        events=('event_count', 'sum'),
        revenue=('total_revenue_usd', 'sum')
    ).unstack(fill_value=0)
    display(pivot)

    print("\n=== Attributed Only: Revenue Status by Date ===")
    df_attr = df_pb_revenue[df_pb_revenue['attributed'] == True]
    if not df_attr.empty:
        display(df_attr[['date', 'revenue_status', 'event_count', 'total_revenue_usd']])
    else:
        print("No attributed revenue events found.")

    print("\n=== Unattributed Only: Revenue Status by Date ===")
    df_unattr = df_pb_revenue[df_pb_revenue['attributed'] == False]
    if not df_unattr.empty:
        display(df_unattr[['date', 'revenue_status', 'event_count', 'total_revenue_usd']])
    else:
        print("No unattributed revenue events found.")

✅ 3b. Raw PB Attributed vs Unattributed: 21 rows


,date,attributed,revenue_status,event_count,total_revenue_usd
0,2026-03-01,True,has_revenue,495,3272.9441
1,2026-03-01,False,has_revenue,4188,26269.0033
2,2026-03-02,True,has_revenue,365,2332.5328
3,2026-03-02,False,has_revenue,3637,18960.1168
4,2026-03-03,True,has_revenue,330,1868.6812
5,2026-03-03,False,has_revenue,2760,14713.2465
6,2026-03-04,True,has_revenue,387,2496.3369
7,2026-03-04,True,zero_revenue,6,0.0000
8,2026-03-04,False,has_revenue,2706,14480.0511
9,2026-03-05,True,has_revenue,271,1584.1682



=== Pivot: Daily Revenue by Attribution Status ===


events           revenue           
attributed  False True        False      True 
date                                          
2026-03-01   4188   495  26269.0033  3272.9441
2026-03-02   3637   365  18960.1168  2332.5328
2026-03-03   2760   330  14713.2465  1868.6812
2026-03-04   2706   393  14480.0511  2496.3369
2026-03-05   2625   281  14880.8253  1584.1682
2026-03-06   2647   325  14915.0478  1925.8928
2026-03-07   2631   330  14857.9604  2017.7435
2026-03-08   3069   411  16852.4495  2309.2538


=== Attributed Only: Revenue Status by Date ===


,date,revenue_status,event_count,total_revenue_usd
0,2026-03-01,has_revenue,495,3272.9441
2,2026-03-02,has_revenue,365,2332.5328
4,2026-03-03,has_revenue,330,1868.6812
6,2026-03-04,has_revenue,387,2496.3369
7,2026-03-04,zero_revenue,6,0.0000
9,2026-03-05,has_revenue,271,1584.1682
10,2026-03-05,zero_revenue,10,0.0000
12,2026-03-06,has_revenue,318,1925.8928
13,2026-03-06,zero_revenue,7,0.0000
15,2026-03-07,has_revenue,309,2017.7435



=== Unattributed Only: Revenue Status by Date ===


,date,revenue_status,event_count,total_revenue_usd
1,2026-03-01,has_revenue,4188,26269.0033
3,2026-03-02,has_revenue,3637,18960.1168
5,2026-03-03,has_revenue,2760,14713.2465
8,2026-03-04,has_revenue,2706,14480.0511
11,2026-03-05,has_revenue,2625,14880.8253
14,2026-03-06,has_revenue,2647,14915.0478
17,2026-03-07,has_revenue,2631,14857.9604
20,2026-03-08,has_revenue,3069,16852.4495


In [82]:
df_pb_by_campaign = run_query("""
SELECT
  DATE(timestamp) AS date,
  moloco.campaign_id,
  CASE
    WHEN event.revenue_usd.amount > 0 THEN 'has_revenue'
    WHEN event.revenue_usd.amount = 0 THEN 'zero_revenue'
    ELSE 'null_revenue'
  END AS revenue_status,
  COUNT(*) AS event_count,
  ROUND(SUM(IFNULL(event.revenue_usd.amount, 0)), 4) AS total_revenue_usd
FROM `focal-elf-631.df_accesslog.pb`
WHERE DATE(timestamp) BETWEEN '2026-03-01' AND '2026-03-08'
  AND app.bundle = 'com.netmarble.sololv'
  AND event.name = 'revenue'
  AND attribution.attributed = true
  AND moloco.campaign_id IN ('Jl5NUKiajhIyKXQK', 'fISFGcvrsN7uXPoR', 'nqWMPgcAgXjZgEM8')
GROUP BY 1, 2, 3
ORDER BY campaign_id, date, revenue_status
""", '3b. Campaign-Level Raw PB')

if not df_pb_by_campaign.empty:
    display(df_pb_by_campaign)

    print("\n=== Pivot: Revenue per Campaign x Date ===")
    pivot_camp = df_pb_by_campaign.groupby(['campaign_id', 'date']).agg(
        events=('event_count', 'sum'),
        revenue=('total_revenue_usd', 'sum')
    ).unstack(fill_value=0)
    display(pivot_camp)

✅ 3b. Campaign-Level Raw PB: 29 rows


,date,campaign_id,revenue_status,event_count,total_revenue_usd
0,2026-03-02,Jl5NUKiajhIyKXQK,has_revenue,2,44.4407
1,2026-03-03,Jl5NUKiajhIyKXQK,has_revenue,4,48.4416
2,2026-03-05,Jl5NUKiajhIyKXQK,zero_revenue,5,0.0000
3,2026-03-06,Jl5NUKiajhIyKXQK,zero_revenue,1,0.0000
4,2026-03-07,Jl5NUKiajhIyKXQK,has_revenue,4,14.1458
5,2026-03-07,Jl5NUKiajhIyKXQK,zero_revenue,3,0.0000
6,2026-03-08,Jl5NUKiajhIyKXQK,has_revenue,2,8.8332
7,2026-03-08,Jl5NUKiajhIyKXQK,zero_revenue,7,0.0000
8,2026-03-02,fISFGcvrsN7uXPoR,has_revenue,1,1.7776
9,2026-03-04,fISFGcvrsN7uXPoR,has_revenue,2,6.2669



=== Pivot: Revenue per Campaign x Date ===


events                                              \
date             2026-03-01 2026-03-02 2026-03-03 2026-03-04 2026-03-05   
campaign_id                                                               
Jl5NUKiajhIyKXQK          0          2          4          0          5   
fISFGcvrsN7uXPoR          0          1          0          2          5   
nqWMPgcAgXjZgEM8          7          1          2          6          5   

                                                     revenue             \
date             2026-03-06 2026-03-07 2026-03-08 2026-03-01 2026-03-02   
campaign_id                                                               
Jl5NUKiajhIyKXQK          1          7          9     0.0000    44.4407   
fISFGcvrsN7uXPoR          1          6         10     0.0000     1.7776   
nqWMPgcAgXjZgEM8         15         19         16    68.6013     0.6060   

                                                                         \
date             2026-03-03 2026-03-04 2026-03-05 2026-03-06 2026-03-07   
campaign_id                                                               
Jl5NUKiajhIyKXQK    48.4416     0.0000     0.0000     0.0000    14.1458   
fISFGcvrsN7uXPoR     0.0000     6.2669     2.3764     3.9075     5.8941   
nqWMPgcAgXjZgEM8     4.5520     0.0000    11.8819    42.9965    22.5329   

                             
date             2026-03-08  
campaign_id                  
Jl5NUKiajhIyKXQK     8.8332  
fISFGcvrsN7uXPoR    22.1418  
nqWMPgcAgXjZgEM8    66.7394

## 3c. Comparison: Raw Postback vs Attributed Table (fact_dsp_all)

Cross-reference raw postback revenue with the attributed table to determine where the gap occurs.

In [83]:
df_attr_daily = run_query("""
SELECT
  DATE(timestamp) AS date,
  COUNT(*) AS attr_revenue_events,
  COUNTIF(cv.revenue_usd.amount > 0) AS attr_with_revenue,
  ROUND(SUM(IFNULL(cv.revenue_usd.amount, 0)), 4) AS attr_revenue_usd
FROM `focal-elf-631.prod_stream_view.cv`
WHERE cv.pb.app.bundle = 'com.netmarble.sololv'
  AND DATE(timestamp) BETWEEN '2026-03-01' AND '2026-03-08'
  AND cv.pb.event.name = 'revenue'
  AND cv.pb.moloco.attributed = true
GROUP BY 1
ORDER BY 1
""", '3c. Attributed Daily Revenue (cv)')

if not df_attr_daily.empty and not df_pb_revenue.empty:
    pb_daily = df_pb_revenue.groupby('date').agg(
        raw_events=('event_count', 'sum'),
        raw_revenue=('total_revenue_usd', 'sum')
    ).reset_index()
    pb_daily['date'] = pb_daily['date'].astype(str)
    df_attr_daily['date'] = df_attr_daily['date'].astype(str)

    comparison = pd.merge(df_attr_daily, pb_daily, on='date', how='outer').sort_values('date')
    comparison['gap'] = comparison['raw_revenue'].fillna(0) - comparison['attr_revenue_usd'].fillna(0)
    print("=== Comparison: Attributed (cv) vs Raw Postback (df_accesslog.pb) ===")
    print("If raw_revenue ≈ attr_revenue_usd → revenue passes through correctly")
    print("If raw_revenue > 0 but attr = 0 → revenue lost during processing")
    print("If both = 0 → MMP never sent revenue\n")
    display(comparison)
elif not df_attr_daily.empty:
    print("=== Attributed Daily Revenue (raw postback query not available) ===")
    display(df_attr_daily)

✅ 3c. Attributed Daily Revenue (cv): 8 rows
=== Comparison: Attributed (cv) vs Raw Postback (df_accesslog.pb) ===
If raw_revenue ≈ attr_revenue_usd → revenue passes through correctly
If raw_revenue > 0 but attr = 0 → revenue lost during processing
If both = 0 → MMP never sent revenue



,date,attr_revenue_events,attr_with_revenue,attr_revenue_usd,raw_events,raw_revenue,gap
0,2026-03-01,275,275,1836.4130,4683,29541.9474,27705.5344
1,2026-03-02,190,190,1241.9172,4002,21292.6496,20050.7324
2,2026-03-03,174,174,858.9420,3090,16581.9277,15722.9857
3,2026-03-04,210,204,1189.8034,3099,16976.3880,15786.5846
4,2026-03-05,133,123,653.2728,2906,16464.9935,15811.7207
5,2026-03-06,191,184,1012.5161,2972,16840.9406,15828.4245
6,2026-03-07,173,152,880.5994,2961,16875.7039,15995.1045
7,2026-03-08,212,190,1109.4819,3480,19161.7033,18052.2214


## 4. Advertiser-Level Analysis — `yfg0At8VksGnt6EO` & `tGYMnmOyEZqjEIND`

Replicates Sections 2–3c for **all campaigns** (spend > 0) under two advertiser IDs.  
Date range: 2026-03-01 to 2026-03-10.

- **Step 4a**: Campaign discovery (spend > 0 only)
- **Step 4b**: Campaign-level attributed revenue events
- **Step 4c**: Raw postback (attributed vs unattributed)
- **Step 4d**: Comparison: cv vs raw postback

In [ ]:
ADVERTISER_IDS = ['yfg0At8VksGnt6EO', 'tGYMnmOyEZqjEIND']
ADV_FILTER = ', '.join(f"'{a}'" for a in ADVERTISER_IDS)

df_campaigns = run_query(f"""
SELECT
  advertiser_id,
  campaign_id,
  ANY_VALUE(campaign.title) AS title,
  ANY_VALUE(campaign.goal) AS goal,
  ANY_VALUE(campaign.os) AS os,
  ANY_VALUE(campaign.country) AS country,
  MIN(date_utc) AS first_date,
  MAX(date_utc) AS last_date,
  ROUND(SUM(gross_spend_usd), 2) AS total_spend,
  SUM(impressions) AS total_impressions,
  SUM(installs) AS total_installs
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE advertiser_id IN ({ADV_FILTER})
  AND date_utc BETWEEN '2026-03-01' AND '2026-03-10'
GROUP BY 1, 2
HAVING total_spend > 0
ORDER BY advertiser_id, total_spend DESC
""", '4a. Campaign Discovery (spend > 0)')

if not df_campaigns.empty:
    for adv_id in ADVERTISER_IDS:
        n = len(df_campaigns[df_campaigns.advertiser_id == adv_id])
        print(f'Advertiser {adv_id}: {n} campaigns with spend')
    display(df_campaigns)

CAMPAIGN_IDS = df_campaigns['campaign_id'].tolist() if not df_campaigns.empty else []
print(f'\nTotal campaigns to analyze: {len(CAMPAIGN_IDS)}')

### 4b. Campaign-Level Attributed Revenue Events

Daily revenue event count and revenue amount per campaign.

In [ ]:
if CAMPAIGN_IDS:
    camp_filter = ', '.join(f"'{c}'" for c in CAMPAIGN_IDS)
    camp_to_adv = df_campaigns.set_index('campaign_id')['advertiser_id'].to_dict()

    df_adv_campaign_rev = run_query(f"""
    SELECT
      cv.pb.moloco.campaign_id AS campaign_id,
      DATE(timestamp) AS date,
      COUNT(*) AS revenue_events,
      COUNTIF(cv.revenue_usd.amount > 0) AS events_with_revenue,
      ROUND(SUM(IFNULL(cv.revenue_usd.amount, 0)), 4) AS pb_revenue
    FROM `focal-elf-631.prod_stream_view.cv`
    WHERE cv.pb.moloco.campaign_id IN ({camp_filter})
      AND DATE(timestamp) BETWEEN '2026-03-01' AND '2026-03-10'
      AND cv.pb.event.name = 'revenue'
      AND cv.pb.moloco.attributed = true
    GROUP BY 1, 2
    ORDER BY campaign_id, date
    """, '4b. Campaign-Level Revenue Events (cv)')

    if not df_adv_campaign_rev.empty:
        df_adv_campaign_rev['advertiser_id'] = df_adv_campaign_rev['campaign_id'].map(camp_to_adv)
        df_adv_campaign_rev['pct_with_revenue'] = (
            df_adv_campaign_rev['events_with_revenue'] / df_adv_campaign_rev['revenue_events'] * 100
        ).round(1)
        df_adv_campaign_rev['flag'] = df_adv_campaign_rev.apply(
            lambda r: 'ALL MISSING' if r['events_with_revenue'] == 0
            else 'PARTIAL' if r['events_with_revenue'] < r['revenue_events']
            else 'OK', axis=1
        )

        for adv_id in ADVERTISER_IDS:
            sub = df_adv_campaign_rev[df_adv_campaign_rev['advertiser_id'] == adv_id]
            if sub.empty:
                print(f'\n{adv_id}: no revenue events found.')
                continue
            print(f'\n=== Advertiser: {adv_id} ===')
            print(f'Campaigns with revenue events: {sub["campaign_id"].nunique()}')

            pivot_rev = sub.pivot_table(
                index='campaign_id', columns='date', values='pb_revenue', aggfunc='sum', fill_value=0
            )
            print('Revenue ($) by Campaign x Date:')
            display(pivot_rev)

            flag_pivot = sub.pivot_table(
                index='campaign_id', columns='date', values='flag', aggfunc='first', fill_value='—'
            )
            print('Flag (OK / PARTIAL / ALL MISSING) by Campaign x Date:')
            display(flag_pivot)
    else:
        df_adv_campaign_rev = pd.DataFrame()
else:
    print('No campaigns found — skipping 4b.')
    df_adv_campaign_rev = pd.DataFrame()

### 4c. Raw Postback — Attributed vs Unattributed Revenue

Check `df_accesslog.pb` before attribution processing.  
If unattributed postbacks also show \$0 → MMP-side issue.  
If only attributed show \$0 → Moloco processing issue.

In [ ]:
if CAMPAIGN_IDS:
    camp_filter = ', '.join(f"'{c}'" for c in CAMPAIGN_IDS)
    camp_to_adv = df_campaigns.set_index('campaign_id')['advertiser_id'].to_dict()

    df_adv_pb = run_query(f"""
    SELECT
      DATE(timestamp) AS date,
      moloco.campaign_id AS campaign_id,
      attribution.attributed,
      CASE
        WHEN event.revenue_usd.amount > 0 THEN 'has_revenue'
        WHEN event.revenue_usd.amount = 0 THEN 'zero_revenue'
        ELSE 'null_revenue'
      END AS revenue_status,
      COUNT(*) AS event_count,
      ROUND(SUM(IFNULL(event.revenue_usd.amount, 0)), 4) AS total_revenue_usd
    FROM `focal-elf-631.df_accesslog.pb`
    WHERE DATE(timestamp) BETWEEN '2026-03-01' AND '2026-03-10'
      AND moloco.campaign_id IN ({camp_filter})
      AND event.name = 'revenue'
    GROUP BY 1, 2, 3, 4
    ORDER BY campaign_id, date, attributed DESC, revenue_status
    """, '4c. Raw PB (Attributed + Unattributed)')

    if not df_adv_pb.empty:
        df_adv_pb['advertiser_id'] = df_adv_pb['campaign_id'].map(camp_to_adv)

        for adv_id in ADVERTISER_IDS:
            sub = df_adv_pb[df_adv_pb['advertiser_id'] == adv_id]
            if sub.empty:
                print(f'\n{adv_id}: no raw postback revenue events found.')
                continue
            print(f'\n=== Advertiser: {adv_id} ===')

            pivot = sub.groupby(['date', 'attributed']).agg(
                events=('event_count', 'sum'),
                revenue=('total_revenue_usd', 'sum')
            ).unstack(fill_value=0)
            print('Daily revenue (attributed vs unattributed):')
            display(pivot)

            attr_sub = sub[(sub['attributed'] == True) & (sub['revenue_status'] == 'zero_revenue')]
            if not attr_sub.empty:
                print('\nAttributed zero-revenue events by campaign x date:')
                display(attr_sub[['date', 'campaign_id', 'event_count', 'total_revenue_usd']])
            else:
                print('\nNo attributed zero-revenue events found.')
    else:
        df_adv_pb = pd.DataFrame()
else:
    print('No campaigns found — skipping 4c.')
    df_adv_pb = pd.DataFrame()

### 4d. Comparison: Attributed (cv) vs Raw Postback

gap = raw\_revenue − attr\_revenue  
Large positive gap → revenue present in raw PB but lost during attribution/processing.

In [ ]:
if CAMPAIGN_IDS and not df_adv_pb.empty and not df_adv_campaign_rev.empty:
    for adv_id in ADVERTISER_IDS:
        print(f'\n=== Advertiser: {adv_id} ===')

        attr = df_adv_campaign_rev[df_adv_campaign_rev['advertiser_id'] == adv_id]
        attr_daily = attr.groupby('date').agg(
            attr_events=('revenue_events', 'sum'),
            attr_with_rev=('events_with_revenue', 'sum'),
            attr_revenue=('pb_revenue', 'sum')
        ).reset_index()
        attr_daily['date'] = attr_daily['date'].astype(str)

        pb = df_adv_pb[df_adv_pb['advertiser_id'] == adv_id]
        pb_daily = pb.groupby('date').agg(
            raw_events=('event_count', 'sum'),
            raw_revenue=('total_revenue_usd', 'sum')
        ).reset_index()
        pb_daily['date'] = pb_daily['date'].astype(str)

        if attr_daily.empty and pb_daily.empty:
            print('  No data found.')
            continue

        comparison = pd.merge(attr_daily, pb_daily, on='date', how='outer').sort_values('date')
        comparison['gap'] = comparison['raw_revenue'].fillna(0) - comparison['attr_revenue'].fillna(0)
        comparison['gap_pct'] = (
            comparison['gap'] / comparison['raw_revenue'].replace(0, float('nan')) * 100
        ).round(1)

        print('Attributed (cv) vs Raw Postback comparison:')
        print('gap = raw_revenue − attr_revenue (positive = revenue dropped after PB receipt)')
        display(comparison[['date', 'attr_events', 'attr_with_rev', 'attr_revenue',
                             'raw_events', 'raw_revenue', 'gap', 'gap_pct']])
else:
    print('Insufficient data to run comparison — check 4b and 4c outputs.')